In [ ]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "data/raw/api_progress.json"
print(path)

In [ ]:
# read all data from path into all_data
with path.open("r") as f:
    all_data = json.load(f)

In [ ]:
import pandas as pd

df = pd.json_normalize(all_data)

In [ ]:
df.head()

In [ ]:
# keep redirect column - we want it after filtering to give to our scraper
df = df.drop(columns=["adref"])
# Drop any column that contains __CLASS__
df = df.drop(columns=[col for col in df.columns if "__CLASS__" in col])

In [ ]:

df['salary_is_predicted'].value_counts()

In [ ]:
[len(x) for x in df['location.area']]
df['location.area_length'] = df['location.area'].apply(len)
df['location.area_length'].value_counts()

# Subset to location length == 5 and pull location.area
# df[df['location.area_length'] == 3]['location.area'].value_counts()


In [ ]:
# create location_1, location_2, location_3 by splitting location.area lists into components, 
# accounting for the fact that not all lists have all 5 components
location_cols = ['location_country', 'location_state', 'location_region', 'location_city', 'location_area', 'location_suburb']
location_df = df['location.area'].apply(lambda x: pd.Series(x + [None] * (6 - len(x))))
location_df.columns = location_cols
df = pd.concat([df, location_df], axis=1)

In [ ]:
df.loc[df['location_city'] == 'Western Sydney', 'location_suburb'].value_counts()

In [ ]:
# df = df.loc[df['salary_min'] > 30000, :]

In [ ]:
df[df['salary_min'] == 0]['salary_max'].value_counts()

In [ ]:
both = df[(df['salary_min'] > 0) & (df['salary_max'] > 0)]
ratio = (both['salary_max'] / both['salary_min']).median()
print(f"Typical max/min ratio: {ratio:.2f}")

In [ ]:
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')
df['missing_long_lat'] = df['longitude'].isna() | df['latitude'].isna()

In [ ]:
df['imputed_salary_min'] = df.apply(lambda row: row['salary_min'] if row['salary_min'] > 0 else row['salary_max'] / ratio, axis=1)
df['imputed_salary_max'] = df.apply(lambda row: row['salary_max'] if row['salary_max'] > 0 else row['salary_min'] * ratio, axis=1)
df['avg_salary'] = (df['imputed_salary_min'] + df['imputed_salary_max']) / 2

In [ ]:
# Coutn unique rows defined by title + description + salary
df['unique_id'] = df['title'].str.lower() + '|' + df['description'] + '|' + df['avg_salary'].astype(str)

# Count how many rows we would have if we dropped duplicates based on unique_id
print(f"Original rows: {len(df)}")
print(f"Unique rows: {df['unique_id'].nunique()}")
df['id'] = df['id'].astype(int)
# Drop rows using unique_id. Keep earliest posting (lowest id)
df = df.sort_values('id').drop_duplicates(subset='unique_id', keep='first').reset_index(drop=True)

In [ ]:


lower = 50_000
upper = 300_000

df = df[df['avg_salary'] >= lower].copy()
df['avg_salary'] = df['avg_salary'].clip(upper=upper)
company_counts = df['company.display_name'].value_counts()
df['company_listing_count'] = df['company.display_name'].map(company_counts)

min_count = 50  # tune this threshold



In [ ]:
min_count = 50
for col in ['location_region', 'location_city']:
    counts = df[col].value_counts()
    # Create new column for the abridged versions of the location 
    df[f'{col}_abridged'] = df[col].where(df[col].map(counts) >= min_count, 'Other')

In [ ]:
# Save the feature engineered data to the /data/clean_data folder as a csv
output_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
df.to_csv(output_path, index=False)


In [ ]:
# Also save just the redirect URLs in JSON format
df['redirect_url'].to_json(PROJECT_ROOT / "data/raw/redirect_urls.json", orient='records')